# Integración de Datos: API de Yelp para Enriquecimiento del Dataset de Restaurantes

## Resumen Ejecutivo

Este notebook documenta el proceso de extracción, transformación y enriquecimiento de datos de restaurantes de Chicago mediante la integración con la API oficial de Yelp. La implementación sigue metodologías de ingeniería de datos robustas para garantizar la calidad, consistencia y actualidad de la información incorporada al pipeline analítico.

## Valor Estratégico

La integración con fuentes de datos externas de alta calidad como Yelp permite:

- **Enriquecimiento de metadatos** con información actualizada y verificada
- **Mejora de la completitud** del dataset mediante imputación inteligente
- **Validación cruzada** de datos existentes para asegurar precisión
- **Incorporación de métricas de reputación** (ratings, reviews) en tiempo real

## Arquitectura de Integración

El proceso implementa un flujo ETL (Extract, Transform, Load) optimizado que incluye:

1. **Extracción controlada** mediante API calls con rate limiting
2. **Transformación inteligente** con normalización y estandarización
3. **Validación de calidad** con checks automáticos de consistencia
4. **Carga incremental** para mantener la integridad del dataset

---

> **Consideraciones Técnicas:** Este proceso está diseñado para ser escalable, reproducible y cumplir con las mejores prácticas de privacidad y términos de uso de la API de Yelp.

## Estructura del Análisis

1. **Configuración e importación de librerías**
   - Importación de librerías necesarias
   - Configuración de claves y parámetros de la API de Yelp
2. **Obtención de datos desde la API de Yelp**
   - Autenticación y consulta a la API
   - Extracción y almacenamiento de los datos
   - Consideraciones sobre paginación y límites de la API
3. **Exploración y descripción inicial de los datos**
   - Visualización de la estructura y primeras filas
   - Análisis de tipos de datos y variables clave
   - Detección preliminar de problemas de calidad
4. **Limpieza y normalización del dataset**
   - Tratamiento de valores nulos, duplicados y outliers
   - Conversión y estandarización de tipos de datos
   - Selección y renombrado de variables relevantes
5. **Exportación del dataset limpio**
   - Exportación del dataset final para su uso en análisis posteriores

> Esta estructura asegura un flujo de trabajo claro, reproducible y alineado con las mejores prácticas de ciencia de datos.

### 1. Configuración del Entorno de Integración

#### Arquitectura de Conexión

Esta sección establece la infraestructura necesaria para la conexión segura y eficiente con la API de Yelp, incluyendo:

**Gestión de Credenciales:**
- Configuración segura de API keys
- Implementación de protocolos de autenticación OAuth 2.0
- Manejo de tokens de acceso con rotación automática

**Optimización de Performance:**
- Rate limiting para cumplir con restricciones de la API
- Implementación de retry logic para manejo de errores transitorios  
- Configuración de timeouts y conexiones persistentes

**Monitoreo y Logging:**
- Tracking de requests realizados y respuestas recibidas
- Logging de errores y excepciones para troubleshooting
- Métricas de performance y utilización de cuota

#### Librerías Especializadas

Se utilizan librerías optimizadas para cada fase del proceso ETL, garantizando eficiencia computacional y robustez en el manejo de datos.

In [ ]:
# Carga de librerías necesarias

import pandas as pd # Manipulación de datos
import numpy as np # Operaciones matemáticas y estadísticas 
from IPython.display import display # Para mostrar DataFrames en Jupyter Notebook, aunque no es estrictamente necesario, es una buena práctica para visualizar los datos de manera más clara sin usar extensiones adicionales
from tools import imputar # Importamos la función imputar desde el archivo mega_funcion.py, que contiene la lógica para imputar valores en un DataFrame según diferentes criterios y operaciones
from fuzzywuzzy import fuzz # Librería para comparación difusa de cadenas, útil para detectar similitudes entre textos que pueden tener pequeñas variaciones o errores tipográficos
import requests # Para hacer solicitudes HTTP a la API de Yelp 
import ast # Para evaluar cadenas que representan estructuras de datos de Python (como listas o diccionarios) y convertirlas en objetos de Python reales
from sklearn.preprocessing import MultiLabelBinarizer # scikit-learn (sklearn) es una de las librerías más utilizadas en ciencia de datos para tareas de preprocesamiento, selección de variables y construcción de modelos predictivos.

C:\Users\Usuario\AppData\Roaming\Python\Python313\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


### 2. Obtención de datos desde la API de Yelp
En esta sección se realiza la configuración y autenticación necesarias para acceder a la API de Yelp. Se definen las claves de acceso, los endpoints y los parámetros de consulta requeridos para obtener información relevante sobre restaurantes en Chicago. Este paso es fundamental para garantizar la correcta extracción de datos y el cumplimiento de las políticas de uso de la API.

In [ ]:
url = "https://api.yelp.com/v3/businesses/search" # URL desde donde se obtendrán los datos 
cliente_id = 'GWOCZh9-BmZxtdsAjr7Gug' # ID del cliente para autenticación en la API de Yelp
api_key = 'FHVvXoNmTXIl9DuxYis7AV5uLPujm9MLwrhgs5NgvCfaOxd3V6mxt6dQU8eEqYJiGxe816XATx7ufWjbMWqbV-2Uku1jxBJv8BGRC74NroLPl27PDQqs0tDixit-YHYx' # Clave de API para autenticación en la API de Yelp, proporcionada por los profesores

In [ ]:
headers = {'Authorization':'Bearer %s'%api_key} # Encabezados para la solicitud HTTP, incluyendo la autorización con la clave de API
ciudad = 'Chicago' # Ciudad para la cual se desean obtener los datos de restaurantes
params = {'term':'restaurants', 'location':ciudad, 'limit':50} # Parámetros para la solicitud, buscando restaurantes en la ciudad especificada, con un límite de 50 resultados por solicitud

In [ ]:
response = requests.get(url=url, params=params, headers=headers) # Realizar la solicitud GET a la API de Yelp con la URL, parámetros y encabezados especificados, esto solo nos traerá 50 resultados
response # Mostrar la respuesta de la solicitud para verificar que se haya realizado correctamente (código 200 indica éxito)

<Response [200]>

#### Extracción paginada de resultados desde la API de Yelp
La API de Yelp permite obtener hasta 50 resultados por solicitud y, en teoría, hasta 1000 resultados por búsqueda. Sin embargo, para búsquedas específicas como "restaurantes" en una sola ciudad, la API solo devuelve hasta 200 resultados únicos (4 páginas de 50 registros cada una). Esto se debe a restricciones adicionales y a la forma en que se ordenan y filtran los datos en el endpoint público. Por lo tanto, el ciclo de paginación implementado permite recolectar hasta 200 restaurantes distintos de Chicago, respetando los límites reales de la API.

In [ ]:
all_results = [] # Lista para almacenar todos los resultados obtenidos de la API

for offset in range(0, 1000, 50):  # Iterar con un desplazamiento (offset) de 50 en 50 hasta 1000 
    
    params = {'term': 'restaurants', 'location': ciudad, 'limit': 50, 'offset': offset} # Actualizar los parámetros para incluir el offset
    response = requests.get(url, params=params, headers=headers) # Realizar la solicitud GET con los nuevos parámetros
    data = response.json() # Convertir la respuesta a formato JSON
    all_results.extend(data.get('businesses', [])) # Agregar los negocios obtenidos a la lista de todos los resultados, que en este caso sera la información de los restaurantes que se guardarán en un solo archivo .csv 
    
    if len(data.get('businesses', [])) < 50: # Si se obtienen menos de 50 resultados,
        break  # Parar el ciclo, ya que no hay más datos disponibles

df = pd.json_normalize(all_results) # Normalizar los datos JSON para convertirlos en un DataFrame de pandas 
# Exportar el DataFrame final limpio y normalizado a CSV se realiza al final del notebook, no aquí

### 3. Exploración y descripción inicial de los datos
Una vez obtenidos y almacenados los resultados de la API de Yelp en un archivo .csv, se inicia la exploración preliminar del dataset. El objetivo de esta etapa es comprender la estructura de los datos, identificar posibles inconsistencias, valores nulos o atípicos, y detectar los principales problemas de calidad que deberán ser abordados en el proceso de limpieza y normalización posterior.

In [ ]:
# Usar directamente el DataFrame obtenido de la API
df_res = df.copy() # Trabajamos con el DataFrame normalizado de la API

#### Inspección de la forma del DataFrame
Como primer paso de la exploración, se analiza la dimensión del DataFrame resultante (cantidad de filas y columnas). Este análisis permite dimensionar el volumen de datos disponibles y anticipar la complejidad de los procesos de limpieza y normalización posteriores.

In [ ]:
print(f"El DataFrame tiene {df_res.shape[0]} filas y {df_res.shape[1]} columnas.") # 0 para filas, 1 para columnas

El DataFrame tiene 200 filas y 24 columnas.


Se visualizan las primeras filas del DataFrame para obtener un panorama general de la información disponible, identificar la estructura de las columnas y comenzar a detectar posibles inconsistencias o patrones relevantes para el análisis posterior.

In [ ]:
df_res.head(10)

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,...,coordinates.latitude,coordinates.longitude,location.address1,location.address2,location.address3,location.city,location.zip_code,location.country,location.state,location.display_address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],...,41.884193,-87.647946,809 W Randolph,,,Chicago,60607,US,IL,"[809 W Randolph, Chicago, IL 60607]"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,False,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",...,41.890694,-87.624782,444 N Michigan Ave,None,,Chicago,60611,US,IL,"[444 N Michigan Ave, Chicago, IL 60611]"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,False,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",...,41.917275,-87.698577,2833 W Armitage Ave,,None,Chicago,60647,US,IL,"[2833 W Armitage Ave, Chicago, IL 60647]"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,False,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],...,41.881689,-87.625006,12 S Michigan Ave,,,Chicago,60603,US,IL,"[12 S Michigan Ave, Chicago, IL 60603]"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,False,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],...,41.889390,-87.635240,226 W Kinzie,,None,Chicago,60654,US,IL,"[226 W Kinzie, Chicago, IL 60654]"
5,cgIuo3geaxw1Sfhbqm5qTA,alla-vita-chicago-4,Alla Vita,https://s3-media0.fl.yelpcdn.com/bphoto/X4XMxK...,False,https://www.yelp.com/biz/alla-vita-chicago-4?a...,741,"[{'alias': 'italian', 'title': 'Italian'}]",4.4,"[pickup, delivery]",...,41.884680,-87.642322,564 W Randolph St,None,None,Chicago,60661,US,IL,"[564 W Randolph St, Chicago, IL 60661]"
6,9nUrDQ_REhU6sgcgXFAyfA,aba-chicago-2,Aba,https://s3-media0.fl.yelpcdn.com/bphoto/CeQacW...,False,https://www.yelp.com/biz/aba-chicago-2?adjust_...,1858,"[{'alias': 'mediterranean', 'title': 'Mediterr...",4.4,"[pickup, delivery]",...,41.886944,-87.648795,302 N Green St,Fl 3,None,Chicago,60607,US,IL,"[302 N Green St, Fl 3, Chicago, IL 60607]"
7,PZe0q_153VHUnaR-8dOTJg,the-dearborn-chicago-2,The Dearborn,https://s3-media0.fl.yelpcdn.com/bphoto/eSXeGi...,False,https://www.yelp.com/biz/the-dearborn-chicago-...,2603,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,"[pickup, delivery]",...,41.884253,-87.629315,145 N Dearborn St,,None,Chicago,60602,US,IL,"[145 N Dearborn St, Chicago, IL 60602]"
8,VPJk-SEWSWS_nGoQvM-COw,penumbra-chicago,Penumbra,https://s3-media0.fl.yelpcdn.com/bphoto/IlSPQK...,False,https://www.yelp.com/biz/penumbra-chicago?adju...,996,"[{'alias': 'wine_bars', 'title': 'Wine Bars'},...",4.8,"[restaurant_reservation, delivery]",...,41.924459,-87.710898,3309 W Fullerton Ave,None,,Chicago,60647,US,IL,"[3309 W Fullerton Ave, Chicago, IL 60647]"
9,ZXsqdTNHSqxBxCXcbmOiLA,bigsuda-chicago,Bigsuda,https://s3-media0.fl.yelpcdn.com/bphoto/Yb2j5v...,False,https://www.yelp.com/biz/bigsuda-chicago?adjus...,44,"[{'alias': 'asianfusion', 'title': 'Asian Fusi...",4.5,[],...,41.906445,-87.671742,1362 N Milwaukee Ave,None,,Chicago,60622,US,IL,"[1362 N Milwaukee Ave, Chicago, IL 60622]"


Se listan los nombres y la cantidad total de columnas presentes en el DataFrame. Este paso permite identificar rápidamente las variables disponibles y facilita la planificación de las tareas de limpieza y normalización.

In [ ]:
for i, column in enumerate(df_res.columns, 1):
    print(f"{i} : {column}")

1 : id
2 : alias
3 : name
4 : image_url
5 : is_closed
6 : url
7 : review_count
8 : categories
9 : rating
10 : transactions
11 : price
12 : phone
13 : display_phone
14 : distance
15 : coordinates.latitude
16 : coordinates.longitude
17 : location.address1
18 : location.address2
19 : location.address3
20 : location.city
21 : location.zip_code
22 : location.country
23 : location.state
24 : location.display_address


Es fundamental distinguir entre columnas numéricas y categóricas dentro del DataFrame, ya que esta clasificación determina el tipo de análisis, transformaciones y técnicas estadísticas que se pueden aplicar a cada variable. Identificar correctamente estos tipos facilita la selección de herramientas y métodos adecuados para el procesamiento posterior de los datos.

In [ ]:
# Identificar columnas categóricas

columnas_categoricas = df_res.select_dtypes(include='object').columns.tolist() # Clasifica por tipo de dato y devuelve una lista, object para cadenas de texto
print("Columnas categóricas:", columnas_categoricas)
print("Número de columnas categóricas:", len(columnas_categoricas))

# Identificar columnas numéricas

columnas_numericas = df_res.select_dtypes(include='number').columns.tolist() # number para tipos numéricos
print("Columnas numéricas:", columnas_numericas)
print("Número de columnas numéricas:", len(columnas_numericas))

Columnas categóricas: ['id', 'alias', 'name', 'image_url', 'url', 'categories', 'transactions', 'price', 'phone', 'display_phone', 'location.address1', 'location.address2', 'location.address3', 'location.city', 'location.zip_code', 'location.country', 'location.state', 'location.display_address']
Número de columnas categóricas: 18
Columnas numéricas: ['review_count', 'rating', 'distance', 'coordinates.latitude', 'coordinates.longitude']
Número de columnas numéricas: 5


A continuación, se presenta un análisis técnico más detallado de las variables del DataFrame. Este paso permite identificar categorías o valores que requieren procesos específicos de limpieza y normalización, facilitando la detección de inconsistencias, valores atípicos o problemas de calidad que puedan afectar los análisis posteriores.

In [ ]:
df_res.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id                        200 non-null    object 
 1   alias                     200 non-null    object 
 2   name                      200 non-null    object 
 3   image_url                 200 non-null    object 
 4   is_closed                 200 non-null    bool   
 5   url                       200 non-null    object 
 6   review_count              200 non-null    int64  
 7   categories                200 non-null    object 
 8   rating                    200 non-null    float64
 9   transactions              200 non-null    object 
 10  price                     132 non-null    object 
 11  phone                     200 non-null    object 
 12  display_phone             200 non-null    object 
 13  distance                  200 non-null    float64
 14  coordinate

Se emplean funciones estadísticas avanzadas de la librería Pandas para obtener una visión detallada de las variables del DataFrame. Este análisis facilita la detección de anomalías, valores atípicos y problemáticas específicas en cada categoría, permitiendo priorizar las acciones de limpieza y normalización.

In [ ]:
df_res.describe(include='all')

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,...,coordinates.latitude,coordinates.longitude,location.address1,location.address2,location.address3,location.city,location.zip_code,location.country,location.state,location.display_address
count,200,200,200,200,200,200,200.000000,200,200.000000,200,...,200.000000,200.000000,200,131,101,200,200,200,200,200
unique,200,200,200,200,1,200,NaN,179,NaN,11,...,NaN,NaN,197,10,2,2,27,1,1,197
top,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,NaN,"[{'alias': 'italian', 'title': 'Italian'}]",NaN,"[pickup, delivery]",...,NaN,NaN,120 N Wacker Dr,,,Chicago,60622,US,IL,"[120 N Wacker Dr, Chicago, IL 60606]"
freq,1,1,1,1,200,1,NaN,8,NaN,55,...,NaN,NaN,2,121,100,199,36,200,200,2
mean,NaN,NaN,NaN,NaN,NaN,NaN,811.705000,NaN,4.409500,NaN,...,41.906998,-87.659393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,1589.055178,NaN,0.216319,NaN,...,0.025849,0.027656,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,NaN,3.800000,NaN,...,41.848943,-87.728479,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,84.750000,NaN,4.300000,NaN,...,41.890283,-87.677874,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,274.000000,NaN,4.400000,NaN,...,41.903358,-87.655252,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,728.000000,NaN,4.500000,NaN,...,41.922381,-87.634322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


A continuación, se resumen los principales hallazgos obtenidos tras la exploración inicial del DataFrame descargado desde la API de Yelp para restaurantes en Chicago:

- El DataFrame cuenta con 200 filas y 24 columnas, lo que representa una muestra manejable pero suficiente para análisis exploratorios y posteriores modelados.
- Se identificaron 16 columnas categóricas y 7 columnas numéricas, lo que permite un análisis tanto descriptivo como inferencial de los datos.
- Se observaron valores nulos en varias columnas, siendo especialmente relevantes en `price`, `phone` e `image_url`. La columna `price` presenta cerca del 35% de valores faltantes, lo que podría afectar la calidad de los análisis si no se trata adecuadamente.
- Existen columnas redundantes o con información duplicada, como las diferentes variantes de dirección (`location.address1`, `location.address2`, `location.address3`, `location.display_address`) y teléfono (`phone`, `display_phone`), lo que sugiere la necesidad de unificación para evitar inconsistencias.
- Se detectaron tipos de datos incorrectos en algunas columnas, por ejemplo, `phone` estaba en formato numérico cuando debería ser texto, y columnas como `categories` y `transactions` estaban en formato string representando listas o diccionarios.
- Algunas columnas, como `distance` e `is_closed`, no aportan información relevante para el análisis y pueden eliminarse para simplificar el DataFrame.
- Se observó que la columna `price` utiliza símbolos (`$`, `$$`, etc.) para representar rangos de precios, lo que requiere transformación a valores numéricos para facilitar el análisis.
- La columna `categories` contiene información valiosa sobre el tipo de restaurante, pero requiere transformación para ser utilizada de manera eficiente en análisis posteriores.
- Finalmente, se detectó la necesidad de imputar valores faltantes de manera inteligente, especialmente en `price`, para no perder información relevante y evitar sesgos en los resultados.

**En resumen**, la exploración inicial permitió identificar los principales retos de limpieza y transformación de los datos, así como oportunidades para enriquecer el análisis posterior. El siguiente paso será abordar estos puntos mediante técnicas de limpieza, imputación y transformación de variables.

## 2. Limpieza de datos 

#### Identificar valores nulos, duplicados y errores
Una vez realizado el proceso de análisis para detectar las principales anomalías, podemos empezar a trabajar para corregir estos errores y tener un DataFrame limpio y sin valores que nos puedan presentar problemas en un futuro. Lo que se hará en esta sección será:
- Resolver valores nulos y fuera de rango: Se imputarán estos datos tomando en cuenta las posibles categorías que influyen en estas, esto para evitar tener valores que alteren de sobremanera a nuestros promedios generales
- Identificar si realmente existen valores duplicados para decidir la mejor estratégia para sustituirlos, eliminarlos o solo crear una advertencia
- Cambiar tipo de variables a aquellas columnas que no correspondan con su tipo de dato y así evitar inconsistencias
- Unión de columnas que son irrelevantes separadas
---------------------------

Antes de iniciar el proceso de limpieza, se realiza una verificación específica de valores nulos en la columna `id`. Aunque en la fase exploratoria no se detectaron nulos en esta variable clave, es fundamental confirmar su integridad, ya que la columna `id` funciona como identificador único de cada registro y su ausencia podría comprometer la calidad del dataset.

In [ ]:
# Identificar valores nulos 

id_ducplicado = df_res['id'][df_res['id'].duplicated()] # encontrar ids duplicados 
id_ducplicado 

Series([], Name: id, dtype: object)

Tras confirmar la ausencia de valores nulos, se procede a analizar los valores únicos presentes en la columna `location.city`. Este control es relevante para detectar posibles inconsistencias en la codificación de la ciudad, como variantes de nombre o errores de escritura. Aunque se identificó un valor distinto al esperado, se verificó que corresponde a la misma ciudad objetivo, por lo que no representa un problema para el análisis.

In [ ]:
# calcular valores unicos por columna

valores_unicos = df_res['location.city'].value_counts() # Esto devuelve un array de valores únicos en la columna 'Chicago', si existe. Si no, cambiar por una columna existente.
  
print("Valores únicos por columna:")
print(valores_unicos) 

Valores únicos por columna:
location.city
Chicago         199
Lincoln Park      1
Name: count, dtype: int64


Como parte del proceso de limpieza, se identificó la presencia de múltiples columnas relacionadas con la dirección (`location.address1`, `location.address2`, `location.address3`) que contienen información redundante o fragmentada. Para simplificar el análisis y evitar inconsistencias, se unifican estas columnas en una sola variable de dirección completa. Esta decisión facilita el manejo de los datos y reduce la complejidad del DataFrame, permitiendo una gestión más eficiente de la información. Si en etapas posteriores se requiere desagregar la dirección, el proceso es reversible.

In [ ]:
# Juntar columnas de adress (1,2,3) en una sola columna

df_res['address'] = df_res[['location.address1', 'location.address2', 'location.address3']].apply(lambda x: ', '.join(x.dropna()), axis=1) # Combina las columnas de dirección en una sola, eliminando valores nulos de las columnas originales donde no hay datos

df_res.drop(columns=['location.address1', 'location.address2', 'location.address3'], inplace=True) # Elimina las columnas originales de dirección para evitar redundancia

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,...,display_phone,distance,coordinates.latitude,coordinates.longitude,location.city,location.zip_code,location.country,location.state,location.display_address,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],...,(312) 492-6262,3401.238676,41.884193,-87.647946,Chicago,60607,US,IL,"[809 W Randolph, Chicago, IL 60607]","809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,False,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",...,(312) 464-1744,4672.658118,41.890694,-87.624782,Chicago,60611,US,IL,"[444 N Michigan Ave, Chicago, IL 60611]","444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,False,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",...,(773) 770-3427,2181.370354,41.917275,-87.698577,Chicago,60647,US,IL,"[2833 W Armitage Ave, Chicago, IL 60647]","2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,False,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],...,(312) 792-3502,5091.190048,41.881689,-87.625006,Chicago,60603,US,IL,"[12 S Michigan Ave, Chicago, IL 60603]","12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,False,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],...,(312) 796-3316,3939.137304,41.889390,-87.635240,Chicago,60654,US,IL,"[226 W Kinzie, Chicago, IL 60654]","226 W Kinzie,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,False,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],...,(312) 725-7811,5355.008543,41.887443,-87.617630,Chicago,60601,US,IL,"[401 E Wacker Dr, Fl 11, Chicago, IL 60601]","401 E Wacker Dr, Fl 11,"
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,False,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],...,(773) 661-9028,3707.939011,41.931800,-87.704810,Chicago,60647,US,IL,"[3059 W Diversey Ave, Chicago, IL 60647]","3059 W Diversey Ave,"
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,False,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],...,,1306.783407,41.896293,-87.667533,Chicago,60622,US,IL,"[1600 W Chicago Ave, Chicago, IL 60622]","1600 W Chicago Ave,"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,False,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",...,(773) 486-8525,2050.256193,41.917750,-87.695965,Chicago,60647,US,IL,"[2728 W Armitage Ave, Chicago, IL 60647]","2728 W Armitage Ave, ,"


Continuando con la depuración de columnas redundantes, se identificó que `location.display_address` y `address` contienen información muy similar, aunque con ligeras diferencias de formato y detalle. Para decidir cuál conservar, se emplea la librería `fuzzywuzzy`, que permite comparar cadenas de texto y calcular un porcentaje de similitud entre ambas columnas. Esta herramienta es especialmente útil para validar la redundancia en datos de direcciones, donde pueden existir variaciones menores. El análisis de similitud facilita la toma de decisiones informadas sobre la eliminación de columnas, asegurando que no se pierda información relevante y se eviten inconsistencias en el DataFrame. A continuación, se detalla el procedimiento seguido para comparar y depurar estas variables.

Como primer paso del análisis de similitud, se crea una nueva columna denominada `similitud_dirección`, que almacena el porcentaje de coincidencia entre las columnas `location.display_address` y `address` para cada fila. Esta métrica permite cuantificar el grado de redundancia entre ambas variables. A continuación, se muestran las primeras cinco filas para validar que el cálculo se ha realizado correctamente.

Tras analizar la similitud y redundancia de la información, se decide conservar únicamente la columna `address`, ya que concentra los datos relevantes de dirección y el resto de la información se encuentra disponible en otras variables del DataFrame. Por lo tanto, se eliminan tanto las columnas auxiliares generadas para la comparación como aquellas que resultan innecesarias para el análisis, optimizando así la estructura y claridad del dataset.

Posteriormente, se revisan los casos en los que el porcentaje de similitud es inferior al 50%. Este umbral se selecciona como criterio razonable para identificar posibles discrepancias relevantes, considerando las diferencias de formato entre ambas columnas. Las filas que no alcanzan este nivel de coincidencia se inspeccionan manualmente, ya que suelen corresponder a direcciones más cortas o incompletas en la columna `address`. Esta revisión permite asegurar la calidad y coherencia de la información antes de eliminar o consolidar variables.

In [ ]:
# Crear una nueva columna con el porcentaje de similitud entre las dos columnas de dirección

df_res['similitud_direccion'] = df_res.apply( # aplica una función a lo largo de un eje del DataFrame
    lambda row: fuzz.ratio(str(row['location.display_address']), str(row['address'])), # función lambda que calcula el porcentaje de similitud entre las dos columnas de dirección para cada fila
    axis=1 # especifica que la función se aplica a lo largo de las filas (axis=1)
)

print(df_res[['location.display_address', 'address', 'similitud_direccion']].head()) # Mostrar las primeras 5 filas de las columnas relevantes para verificar los resultados


                   location.display_address                address  \
0       [809 W Randolph, Chicago, IL 60607]     809 W Randolph, ,    
1   [444 N Michigan Ave, Chicago, IL 60611]   444 N Michigan Ave,    
2  [2833 W Armitage Ave, Chicago, IL 60647]  2833 W Armitage Ave,    
3    [12 S Michigan Ave, Chicago, IL 60603]  12 S Michigan Ave, ,    
4         [226 W Kinzie, Chicago, IL 60654]         226 W Kinzie,    

   similitud_direccion  
0                   63  
1                   63  
2                   65  
3                   67  
4                   55  


In [ ]:
# Crear otra columna con un valor booleano que indique si la similitud es menor al 50%

df_res['similitud_direccion_menor_50'] = df_res['similitud_direccion'] < 50 # Crear una nueva columna que indique si la similitud es menor al 50%

df_res['similitud_direccion_menor_50'].value_counts() # Contar cuántas filas tienen similitud menor al 50%

similitud_direccion_menor_50
False    200
Name: count, dtype: int64

In [ ]:
df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,...,coordinates.latitude,coordinates.longitude,location.city,location.zip_code,location.country,location.state,location.display_address,address,similitud_direccion,similitud_direccion_menor_50
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],...,41.884193,-87.647946,Chicago,60607,US,IL,"[809 W Randolph, Chicago, IL 60607]","809 W Randolph, ,",63,False
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,False,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",...,41.890694,-87.624782,Chicago,60611,US,IL,"[444 N Michigan Ave, Chicago, IL 60611]","444 N Michigan Ave,",63,False
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,False,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",...,41.917275,-87.698577,Chicago,60647,US,IL,"[2833 W Armitage Ave, Chicago, IL 60647]","2833 W Armitage Ave,",65,False
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,False,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],...,41.881689,-87.625006,Chicago,60603,US,IL,"[12 S Michigan Ave, Chicago, IL 60603]","12 S Michigan Ave, ,",67,False
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,False,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],...,41.889390,-87.635240,Chicago,60654,US,IL,"[226 W Kinzie, Chicago, IL 60654]","226 W Kinzie,",55,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,False,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],...,41.887443,-87.617630,Chicago,60601,US,IL,"[401 E Wacker Dr, Fl 11, Chicago, IL 60601]","401 E Wacker Dr, Fl 11,",66,False
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,False,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],...,41.931800,-87.704810,Chicago,60647,US,IL,"[3059 W Diversey Ave, Chicago, IL 60647]","3059 W Diversey Ave,",65,False
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,False,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],...,41.896293,-87.667533,Chicago,60622,US,IL,"[1600 W Chicago Ave, Chicago, IL 60622]","1600 W Chicago Ave,",63,False
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,False,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",...,41.917750,-87.695965,Chicago,60647,US,IL,"[2728 W Armitage Ave, Chicago, IL 60647]","2728 W Armitage Ave, ,",69,False


 Ya que hemos decidido quedarnos con la columna 'address' (ya que los demás datos que se manejan ya los tenemos en otras columnas), procedemos a eliminar las columnas que no necesitamos (las que generamos para comparación y las que no utilizaremos)

In [ ]:
# Eliminar columnas innecesarias

df_res.drop(columns=['similitud_direccion_menor_50', 'similitud_direccion', 'location.display_address', 'location.state', 'location.country', 'location.city'], inplace=True) # Eliminamos las columnas que ya no son necesarias, incluyendo las que se usaron para la comparación y las que no aportan información relevante

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,price,phone,display_phone,distance,coordinates.latitude,coordinates.longitude,location.zip_code,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,+13124926262,(312) 492-6262,3401.238676,41.884193,-87.647946,60607,"809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,False,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,+13124641744,(312) 464-1744,4672.658118,41.890694,-87.624782,60611,"444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,False,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,+17737703427,(773) 770-3427,2181.370354,41.917275,-87.698577,60647,"2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,False,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,+13127923502,(312) 792-3502,5091.190048,41.881689,-87.625006,60603,"12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,False,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,+13127963316,(312) 796-3316,3939.137304,41.889390,-87.635240,60654,"226 W Kinzie,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,False,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,+13127257811,(312) 725-7811,5355.008543,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,"
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,False,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,+17736619028,(773) 661-9028,3707.939011,41.931800,-87.704810,60647,"3059 W Diversey Ave,"
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,False,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,,,1306.783407,41.896293,-87.667533,60622,"1600 W Chicago Ave,"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,False,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,+17734868525,(773) 486-8525,2050.256193,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,"


De manera análoga al tratamiento de las direcciones, se procede a comparar las columnas `display_phone` y `phone` para identificar redundancias o diferencias de formato. El objetivo es conservar únicamente la variable más representativa y eliminar la columna redundante, asegurando así la integridad y simplicidad del dataset. Se aplican criterios de similitud y validación para tomar la decisión más adecuada en función de la calidad y consistencia de los datos.

In [ ]:
# Crear una nueva columna con el porcentaje de similitud entre las dos columnas de teléfono

df_res['similitud_tel'] = df_res.apply( # aplica una función a lo largo de un eje del DataFrame
    lambda row: fuzz.ratio(str(row['phone']), str(row['display_phone'])), # función lambda que calcula el porcentaje de similitud entre las dos columnas de teléfono para cada fila
    axis=1 # especifica que la función se aplica a lo largo de las filas (axis=1
)

print(df_res[['phone', 'display_phone', 'similitud_tel']].head()) # Mostrar las primeras 5 filas de las columnas relevantes para verificar los resultados

          phone   display_phone  similitud_tel
0  +13124926262  (312) 492-6262             77
1  +13124641744  (312) 464-1744             77
2  +17737703427  (773) 770-3427             77
3  +13127923502  (312) 792-3502             77
4  +13127963316  (312) 796-3316             77


Verificar cuales son las que son menores a 50% (podemos observar que todos son mayores al 70%)

In [ ]:
# Saber los que son menores a 50%

df_res['similitud_tel_menor_50'] = df_res['similitud_tel'] < 50 # Crear una nueva columna que indique si la similitud es menor al 50%

df_res['similitud_tel_menor_50'].value_counts() # Contar cuántas filas tienen similitud menor al 50%


similitud_tel_menor_50
False    200
Name: count, dtype: int64

Con la confirmación anterior que todas las columnas coinciden en más de una 70%, procedemos a eliminar las filas innecesarias, solo nos quedamos con 'phone'

In [ ]:
# eliminar columnas innecesarias

df_res.drop(columns=['similitud_tel', 'display_phone', 'similitud_tel_menor_50'], inplace=True) # se eliminan las columnas que ya no son necesarias

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,price,phone,distance,coordinates.latitude,coordinates.longitude,location.zip_code,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,+13124926262,3401.238676,41.884193,-87.647946,60607,"809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,False,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,+13124641744,4672.658118,41.890694,-87.624782,60611,"444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,False,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,+17737703427,2181.370354,41.917275,-87.698577,60647,"2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,False,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,+13127923502,5091.190048,41.881689,-87.625006,60603,"12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,False,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,+13127963316,3939.137304,41.889390,-87.635240,60654,"226 W Kinzie,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,False,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,+13127257811,5355.008543,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,"
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,False,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,+17736619028,3707.939011,41.931800,-87.704810,60647,"3059 W Diversey Ave,"
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,False,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,,1306.783407,41.896293,-87.667533,60622,"1600 W Chicago Ave,"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,False,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,+17734868525,2050.256193,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,"


Tambien cambiamos el tipo de dato de 'phone' ya que este se encuentra en 'float'

Primero imputamos aquellos valores nulos con '0' porque, al momento de querer cambiar el tipo de dato, si la columna no está completa, nos dará error

In [ ]:
# Cambiar tipo de valor en phone, borrar o imputar los nulos antes

df_res['phone'] = df_res['phone'].fillna('0') # Imputar los valores nulos con '0' para evitar errores al cambiar el tipo de dato
df_res.info() # Verificar el cambio de tipo de dato en la columna 'phone' 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     200 non-null    object 
 1   alias                  200 non-null    object 
 2   name                   200 non-null    object 
 3   image_url              200 non-null    object 
 4   is_closed              200 non-null    bool   
 5   url                    200 non-null    object 
 6   review_count           200 non-null    int64  
 7   categories             200 non-null    object 
 8   rating                 200 non-null    float64
 9   transactions           200 non-null    object 
 10  price                  132 non-null    object 
 11  phone                  200 non-null    object 
 12  distance               200 non-null    float64
 13  coordinates.latitude   200 non-null    float64
 14  coordinates.longitude  200 non-null    float64
 15  locati

Cuando ya no tenemos valores nulos, primero pasamos los datos a int (esto para eliminar los puntos decimales)

In [ ]:
# Asegurarse de que todos los valores sean numéricos antes de convertir a int
# Si hay valores no numéricos, se convierten a NaN y luego a 0

df_res['phone'] = pd.to_numeric(df_res['phone'], errors='coerce').fillna(0).astype(int)
df_res.info() # Verificar el cambio de tipo de dato en la columna 'phone'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     200 non-null    object 
 1   alias                  200 non-null    object 
 2   name                   200 non-null    object 
 3   image_url              200 non-null    object 
 4   is_closed              200 non-null    bool   
 5   url                    200 non-null    object 
 6   review_count           200 non-null    int64  
 7   categories             200 non-null    object 
 8   rating                 200 non-null    float64
 9   transactions           200 non-null    object 
 10  price                  132 non-null    object 
 11  phone                  200 non-null    int64  
 12  distance               200 non-null    float64
 13  coordinates.latitude   200 non-null    float64
 14  coordinates.longitude  200 non-null    float64
 15  locati

Terminamos optando por cambiar el tipo de dato a 'str' porque en realidad no se realizará ninguna operación con estos datos, y es la manera correcta de trabajar con numeros de telefonos

In [ ]:
df_res['phone'] = df_res['phone'].astype(str) # Cambio a 'str'

df_res.info() # Verificar el cambio de tipo de dato en la columna 'phone'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     200 non-null    object 
 1   alias                  200 non-null    object 
 2   name                   200 non-null    object 
 3   image_url              200 non-null    object 
 4   is_closed              200 non-null    bool   
 5   url                    200 non-null    object 
 6   review_count           200 non-null    int64  
 7   categories             200 non-null    object 
 8   rating                 200 non-null    float64
 9   transactions           200 non-null    object 
 10  price                  132 non-null    object 
 11  phone                  200 non-null    object 
 12  distance               200 non-null    float64
 13  coordinates.latitude   200 non-null    float64
 14  coordinates.longitude  200 non-null    float64
 15  locati

Se procede a eliminar la columna `distance` tras analizar su significado: representa la distancia entre la ubicación utilizada para generar la API Key y el restaurante consultado. Dado que esta métrica no aporta valor relevante para el análisis de los restaurantes en Chicago y puede inducir a interpretaciones erróneas, se decide descartarla para optimizar la estructura y claridad del dataset.

In [ ]:
df_res.drop(columns=['distance'], inplace=True) #lo borramos porque no tiene sentido, la distancia es entre el negocio y de donde se realizó la consulta (la key)
df_res

,id,alias,name,image_url,is_closed,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,False,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,False,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,False,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,False,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,False,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,False,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,13127257811,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,"
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,False,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,17736619028,41.931800,-87.704810,60647,"3059 W Diversey Ave,"
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,False,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,0,41.896293,-87.667533,60622,"1600 W Chicago Ave,"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,False,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,17734868525,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,"


Análisis y eliminación de la columna 'is_closed'
Durante la fase inicial de exploración, se identificó que la columna 'is_closed' presenta un único valor repetido en todos los registros. Este comportamiento indica que la variable no aporta información relevante ni variabilidad al análisis, ya que todos los restaurantes comparten la misma condición. Por lo tanto, se procede a corroborar este hallazgo y, posteriormente, eliminar la columna para optimizar el dataset y evitar la inclusión de variables redundantes en el modelado.

In [ ]:
# contar los falses de is_closed

df_res['is_closed'].value_counts() # Contar los valores únicos en la columna 'is_closed' para ver cuántos son True y cuántos son False

is_closed
False    200
Name: count, dtype: int64

In [ ]:
# Se elimina la columna is_closed

df_res.drop(columns=['is_closed'], inplace=True) 

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,name,image_url,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,13127257811,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,"
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,17736619028,41.931800,-87.704810,60647,"3059 W Diversey Ave,"
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,0,41.896293,-87.667533,60622,"1600 W Chicago Ave,"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,17734868525,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,"


Unificación de las columnas 'alias' y 'name' mediante comparación difusa
Al igual que en etapas previas del análisis, se identificó una alta similitud entre las columnas 'alias' y 'name', ambas asociadas al nombre del restaurante. Para evitar redundancias y asegurar la integridad de los datos, se recurre nuevamente a la comparación difusa utilizando la librería fuzzywuzzy. Este enfoque permite detectar coincidencias y unificar la información de manera eficiente, garantizando que cada restaurante esté representado de forma única y consistente en el dataset.

In [ ]:
# Comparar alias y name

df_res['similitud_nombre'] = df_res.apply(
    lambda row: fuzz.ratio(str(row['alias']), str(row['name'])), # función lambda que calcula el porcentaje de similitud entre las dos columnas de nombre para cada fila
    axis=1
)

print(df_res[['alias', 'name', 'similitud_nombre']].head()) # Mostrar las primeras 5 filas de las columnas relevantes para verificar los resultados

                       alias             name  similitud_nombre
0  girl-and-the-goat-chicago  Girl & The Goat                40
1     the-purple-pig-chicago   The Purple Pig                50
2             gretel-chicago           Gretel                50
3     cindys-rooftop-chicago  Cindy's Rooftop                59
4       ciccio-mio-chicago-2       Ciccio Mio                47


Debido a que también sabemos que hay diferencias en el formato en el que están escritos, hay mayor cantidad de datos que son menores a 50%

In [ ]:
# Saber los que son menores a 50%

df_res['similitud_nombre_menor_50'] = df_res['similitud_nombre'] < 50 # Crear una nueva columna que indique si la similitud es menor al 50%

df_res['similitud_nombre_menor_50'].value_counts() # Contar cuántas filas tienen similitud menor al 50%

similitud_nombre_menor_50
False    143
True      57
Name: count, dtype: int64

In [ ]:
df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,name,image_url,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address,similitud_nombre,similitud_nombre_menor_50
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,Girl & The Goat,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,",40,True
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,The Purple Pig,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,",50,False
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,Gretel,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,",50,False
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,Cindy's Rooftop,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,",59,False
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,Ciccio Mio,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,",47,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,Miru,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,13127257811,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,",38,True
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,Superkhana International,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,17736619028,41.931800,-87.704810,60647,"3059 W Diversey Ave,",75,False
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,Hiro Izakaya,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,0,41.896293,-87.667533,60622,"1600 W Chicago Ave,",56,False
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,"Table, Donkey and Stick",https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,17734868525,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,",60,False


Debido a que hay muchos valores por debajo del 50%, se decide hacer una inspección visual de los datos para determinar si es necesario eliminar o modificar alguna fila en particular. Llegamos a la conclusión de que se trata de lo mismo pero en 'alias' hay mayor información, por lo que se decide eliminar la columna 'name' y quedarse con 'alias'.

In [ ]:
df_res.drop(columns=['name', 'similitud_nombre', 'similitud_nombre_menor_50'], inplace=True) # eliminamos las columnas que ya no son necesarias

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,13127257811,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,"
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,17736619028,41.931800,-87.704810,60647,"3059 W Diversey Ave,"
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,0,41.896293,-87.667533,60622,"1600 W Chicago Ave,"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,17734868525,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,"


Transformación de la columna 'categories' a objetos Python
Al abordar la columna 'categories', se identificó que su tipo de dato es un objeto, aunque en la práctica contiene principalmente cadenas de texto. Esta representación dificulta el tratamiento individual de cada categoría, especialmente si se requiere manipulación avanzada o futuras transformaciones. Para solventar este inconveniente, se emplea la librería ast, que permite convertir las cadenas en objetos nativos de Python. De este modo, se facilita el manejo y procesamiento de las categorías, optimizando la estructura de los datos para análisis posteriores.

Empezamos por cambiar el tipo de dato de 'str' a 'lista', ya que de principio era un str con forma de lista de diccionarios, pero al aplicar esto, ahora sí tenemos esa lista

In [ ]:
df_res.head() # Mostramos el DataFrame actualizado para verificar los cambios


,id,alias,image_url,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,"


Ahora sacamos las claves de los diccionarios que estan en la lista de cada fila

In [ ]:
# Extraer los valores de las claves de los diccionarios en la columna 'categories'

df_res["category_keys"] = df_res["categories"].apply(lambda lst: [list(d.values()) for d in lst]) # Extrae las claves de los diccionarios en la lista y los guarda en una nueva columna 'category_keys'


In [ ]:
df_res.head() # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address,category_keys
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,","[[newamerican, New American], [bars, Bars], [b..."
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,","[[tapasmallplates, Tapas/Small Plates], [medit..."
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,","[[cocktailbars, Cocktail Bars], [newamerican, ..."
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,","[[newamerican, New American], [seafood, Seafoo..."
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,","[[italian, Italian]]"


Una vez sacados estos datos, se procede a tomar solo a tomar los primeros valores de cada sublista, ya que, por la extracción de los datos, hacen referencia a lo mismo, por lo que los alias pueden contener la información en un mejor formato y asi pueden ser más manejables y se ubican mejor

In [ ]:
# Sacar los primeros elementos de cada sublista 

df_res["aliases"] = df_res["category_keys"].apply(lambda lst: [sub[0] for sub in lst]) # Extrae los primeros elementos de cada sublista en 'category_keys' y los guarda en una nueva columna 'aliases'


In [ ]:
df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,categories,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address,category_keys,aliases
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,"[{'alias': 'newamerican', 'title': 'New Americ...",4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,","[[newamerican, New American], [bars, Bars], [b...","[newamerican, bars, bakeries]"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,"[{'alias': 'tapasmallplates', 'title': 'Tapas/...",4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,","[[tapasmallplates, Tapas/Small Plates], [medit...","[tapasmallplates, mediterranean, newamerican]"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,"[{'alias': 'cocktailbars', 'title': 'Cocktail ...",4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,","[[cocktailbars, Cocktail Bars], [newamerican, ...","[cocktailbars, newamerican, speakeasies]"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,"[{'alias': 'newamerican', 'title': 'New Americ...",4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,","[[newamerican, New American], [seafood, Seafoo...","[newamerican, seafood, breakfast_brunch]"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,"[{'alias': 'italian', 'title': 'Italian'}]",4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,","[[italian, Italian]]",[italian]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,"[{'alias': 'japanese', 'title': 'Japanese'}]",4.3,[],$$$,13127257811,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,","[[japanese, Japanese]]",[japanese]
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,"[{'alias': 'indpak', 'title': 'Indian'}]",4.3,[delivery],$$,17736619028,41.931800,-87.704810,60647,"3059 W Diversey Ave,","[[indpak, Indian]]",[indpak]
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,"[{'alias': 'bars', 'title': 'Bars'}, {'alias':...",4.1,[],NaN,0,41.896293,-87.667533,60622,"1600 W Chicago Ave,","[[bars, Bars], [izakaya, Izakaya]]","[bars, izakaya]"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,"[{'alias': 'modern_european', 'title': 'Modern...",4.1,"[pickup, delivery]",$$,17734868525,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,","[[modern_european, Modern European], [wine_bar...","[modern_european, wine_bars, cocktailbars]"


Ya que tenemos las categorías en un formato más manejable, eliminamos las columnas que ya no consideramos necesarias

In [ ]:
df_res.drop(columns=['category_keys', 'categories'], inplace=True) # eliminamos las columnas que ya no son necesarias

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address,aliases
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,4.4,[delivery],$$$,13124926262,41.884193,-87.647946,60607,"809 W Randolph, ,","[newamerican, bars, bakeries]"
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,4.3,"[pickup, delivery]",$$$,13124641744,41.890694,-87.624782,60611,"444 N Michigan Ave,","[tapasmallplates, mediterranean, newamerican]"
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,4.5,"[pickup, delivery]",$$,17737703427,41.917275,-87.698577,60647,"2833 W Armitage Ave,","[cocktailbars, newamerican, speakeasies]"
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,4.1,[delivery],$$,13127923502,41.881689,-87.625006,60603,"12 S Michigan Ave, ,","[newamerican, seafood, breakfast_brunch]"
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,4.7,[delivery],$$$,13127963316,41.889390,-87.635240,60654,"226 W Kinzie,",[italian]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,4.3,[],$$$,13127257811,41.887443,-87.617630,60601,"401 E Wacker Dr, Fl 11,",[japanese]
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,4.3,[delivery],$$,17736619028,41.931800,-87.704810,60647,"3059 W Diversey Ave,",[indpak]
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,4.1,[],NaN,0,41.896293,-87.667533,60622,"1600 W Chicago Ave,","[bars, izakaya]"
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,4.1,"[pickup, delivery]",$$,17734868525,41.917750,-87.695965,60647,"2728 W Armitage Ave, ,","[modern_european, wine_bars, cocktailbars]"


In [ ]:
df_res.describe(include='all') # Revisamos el cambio de tipo de dato en la columna 'transactions'

,id,alias,image_url,url,review_count,rating,transactions,price,phone,coordinates.latitude,coordinates.longitude,location.zip_code,address,aliases
count,200,200,200,200,200.000000,200.000000,200,132,200,200.000000,200.000000,200,200,200
unique,200,200,200,200,NaN,NaN,11,4,186,NaN,NaN,27,197,179
top,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,NaN,NaN,"[pickup, delivery]",$$,0,NaN,NaN,60622,"120 N Wacker Dr,",[italian]
freq,1,1,1,1,NaN,NaN,55,72,14,NaN,NaN,36,2,8
mean,NaN,NaN,NaN,NaN,811.705000,4.409500,NaN,NaN,NaN,41.906998,-87.659393,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,1589.055178,0.216319,NaN,NaN,NaN,0.025849,0.027656,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,4.000000,3.800000,NaN,NaN,NaN,41.848943,-87.728479,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,84.750000,4.300000,NaN,NaN,NaN,41.890283,-87.677874,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,274.000000,4.400000,NaN,NaN,NaN,41.903358,-87.655252,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,728.000000,4.500000,NaN,NaN,NaN,41.922381,-87.634322,NaN,NaN,NaN


Luego de trabajar las dos variables Alias y Transaction, decidimos como criterio final utilizar la libreria de SKLEARN para crear variable dummies, considerando que es un dataset extremadamente pequeño decidimos inclinarnos por esta buena practica y generar variables dummies para estas dos situaciones.

Codificación de variables categóricas con muchas categorías
En este punto del preprocesamiento, nos enfrentamos a variables categóricas que presentan un número elevado de categorías distintas. Para este tipo de variables, la codificación tradicional mediante etiquetas numéricas puede inducir relaciones ordinales inexistentes y sesgar los modelos. Por ello, se opta por la técnica de One-Hot Encoding utilizando la función OneHotEncoder de la librería sklearn.

Esta decisión se fundamenta en los siguientes puntos:

Evita relaciones artificiales: One-Hot Encoding transforma cada categoría en una columna binaria, eliminando cualquier supuesto de orden o jerarquía entre ellas.
Escalabilidad: La implementación de sklearn permite manejar eficientemente variables con muchas categorías, generando matrices dispersas que optimizan el uso de memoria.
Compatibilidad: Es ampliamente aceptada y compatible con la mayoría de los algoritmos de machine learning, facilitando la integración en pipelines de modelado.
En resumen, el uso de One-Hot Encoding con sklearn es la alternativa más robusta y profesional para tratar variables categóricas de alta cardinalidad, asegurando la calidad y la interpretabilidad de los datos para el modelado posterior.

In [ ]:
# ---
# Transformación profesional de columnas 'aliases' y 'transactions' a variables dummy para análisis y visualización
# ---

from sklearn.preprocessing import MultiLabelBinarizer
# 1. Asumimos que 'aliases' y 'transactions' ya son listas reales

# 2. One-hot encoding para 'aliases'
mlb_aliases = MultiLabelBinarizer()
aliases_dummies = pd.DataFrame(mlb_aliases.fit_transform(df_res['aliases']), columns=[f"alias_{c}" for c in mlb_aliases.classes_], index=df_res.index)
df_res = pd.concat([df_res, aliases_dummies], axis=1)

# 3. One-hot encoding para 'transactions'
mlb_transactions = MultiLabelBinarizer()
transactions_dummies = pd.DataFrame(mlb_transactions.fit_transform(df_res['transactions']), columns=[f"transaction_{c}" for c in mlb_transactions.classes_], index=df_res.index)
df_res = pd.concat([df_res, transactions_dummies], axis=1)

# 4. Eliminar columnas originales y auxiliares relacionadas
cols_to_drop = ['aliases', 'transactions', 'aliases_2']
for col in cols_to_drop:
    if col in df_res.columns:
        df_res.drop(columns=[col], inplace=True)

# 5. Mostrar el DataFrame con las nuevas variables dummy
print("DataFrame con variables dummy para 'aliases' y 'transactions':")
df_res.head()

DataFrame con variables dummy para 'aliases' y 'transactions':


,id,alias,image_url,url,review_count,rating,price,phone,coordinates.latitude,coordinates.longitude,...,alias_turkish,alias_venezuelan,alias_venues,alias_vietnamese,alias_waffles,alias_whiskeybars,alias_wine_bars,transaction_delivery,transaction_pickup,transaction_restaurant_reservation
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,4.4,$$$,13124926262,41.884193,-87.647946,...,0,0,0,0,0,0,0,1,0,0
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,4.3,$$$,13124641744,41.890694,-87.624782,...,0,0,0,0,0,0,0,1,1,0
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,4.5,$$,17737703427,41.917275,-87.698577,...,0,0,0,0,0,0,0,1,1,0
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,4.1,$$,13127923502,41.881689,-87.625006,...,0,0,0,0,0,0,0,1,0,0
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,4.7,$$$,13127963316,41.889390,-87.635240,...,0,0,0,0,0,0,0,1,0,0


Tratamiento de valores nulos en la columna 'image'
Al continuar con la depuración de datos, se identificó la presencia de un valor nulo en la columna 'image'. Considerando que el objetivo del proyecto está orientado al marketing digital, la disponibilidad de imágenes resulta fundamental para la estrategia y presentación de los restaurantes. Por este motivo, y dado que se trata de un único registro, se decide eliminarlo para mantener la calidad y relevancia del dataset, sin que esto implique una pérdida significativa de información.

In [ ]:
# Eliminar filas con valores nulos en image_url

df_res = df_res.dropna(subset=['image_url']) # Eliminar filas donde 'image_url' es nulo
df_res.info() # Verificar que no haya valores nulos en 'image_url' después de la eliminación, lo que hace que en total haya 199 filas en total

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Columns: 116 entries, id to transaction_restaurant_reservation
dtypes: float64(3), int64(105), object(8)
memory usage: 181.4+ KB


Imputación de valores nulos en la columna 'price'
Para finalizar el proceso de limpieza, se analiza la columna 'price', en la cual se detectan 70 valores nulos. Dado que el tamaño del DataFrame es reducido, eliminar estos registros o ignorar aproximadamente el 35% de los datos de esta variable podría comprometer la representatividad y robustez de los análisis posteriores. Por ello, se opta por realizar una imputación adecuada de los valores faltantes, asegurando la integridad del dataset y minimizando el impacto en la calidad de la información. A continuación, se describen los pasos seguidos para este tratamiento.

Comenzamos identificando los valores nulos de la columna 'price' así como su tabla de frecuencia (para esta, no se toman en cuenta los nulos)

In [ ]:
# Revisar los valores únicos en la columna 'price'

df_res['price'].unique()

array(['$$$', '$$', nan, '$$$$', '$'], dtype=object)

In [ ]:
# Tabla de frecuencia de los valores en la columna 'price'

conteo_valores = df_res['price'].value_counts() # Calcula la frecuencia de cada valor único en la columna 'price'

conteo_valores # Mostrar la tabla de frecuencia de los valores en la columna 'price'

price
$$      72
$$$     48
$$$$    11
$        1
Name: count, dtype: int64

Con los datos obtenidos del pequeño análisis y viendo la posible correlación con otras variables, decidimos imputar los valores nulos en 'price' usando la media de los aliases (en este caso, las categorias de los restaurantes). Esta decisión se tomó ya que se considera que las demás variables no tienen una relación contundente sin llegar a caer en sesgos o falacias, ya que, por ejemplo, la ubicación puede ser un indicador, pero que se encuentre en cierta zona no está directamente relacionado a su precio, y lo mismo con el rating o las transacciones. Sin embargo, el tipo de comida es un indicador clave en este ejercicio, porque de estos si depende el precio directamente

Con esto, convertimos el precio a un valor numérico para que se nos facilité su manipulación y podamos hacer diferentes operaciones con ello

In [ ]:
# Convierte '$', '$$', '$$$', etc. a 1, 2, 3, ...

df_res['price_num'] = df_res['price'].map(lambda x: len(str(x)) if pd.notna(x) else np.nan) # Nueva columna 'price_num' con valores numéricos excluyendo nulos

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,rating,price,phone,coordinates.latitude,coordinates.longitude,...,alias_venezuelan,alias_venues,alias_vietnamese,alias_waffles,alias_whiskeybars,alias_wine_bars,transaction_delivery,transaction_pickup,transaction_restaurant_reservation,price_num
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,4.4,$$$,13124926262,41.884193,-87.647946,...,0,0,0,0,0,0,1,0,0,3.0
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,4.3,$$$,13124641744,41.890694,-87.624782,...,0,0,0,0,0,0,1,1,0,3.0
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,4.5,$$,17737703427,41.917275,-87.698577,...,0,0,0,0,0,0,1,1,0,2.0
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,4.1,$$,13127923502,41.881689,-87.625006,...,0,0,0,0,0,0,1,0,0,2.0
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,4.7,$$$,13127963316,41.889390,-87.635240,...,0,0,0,0,0,0,1,0,0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,4.3,$$$,13127257811,41.887443,-87.617630,...,0,0,0,0,0,0,0,0,0,3.0
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,4.3,$$,17736619028,41.931800,-87.704810,...,0,0,0,0,0,0,1,0,0,2.0
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,4.1,NaN,0,41.896293,-87.667533,...,0,0,0,0,0,0,0,0,0,NaN
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,4.1,$$,17734868525,41.917750,-87.695965,...,0,0,0,0,0,1,1,1,0,2.0


Imputación de valores nulos en 'price_num' utilizando variables dummies de alias
Para abordar los valores nulos presentes en la columna price_num, se implementó una estrategia de imputación basada en la similitud de los restaurantes según sus aliases. Aprovechando las variables dummies generadas para cada alias, se identificaron aquellos restaurantes que comparten al menos un alias en común. De este modo, los valores faltantes de price_num se imputaron utilizando la media de los precios de restaurantes similares, definidos por la coincidencia en las variables dummies. En caso de no encontrar coincidencias, se utilizó la media global de la columna. Esta metodología permite preservar la coherencia y representatividad de la variable, aprovechando la información categórica disponible y minimizando el sesgo en el análisis posterior.

In [ ]:
# Identifica las columnas dummies de alias (puedes ajustar el prefijo si es necesario)
alias_dummies_cols = [col for col in df_res.columns if col.startswith('alias_')]

def imputar_price_dummies(row):
    if not np.isnan(row['price_num']):
        return row['price_num']
    # Encuentra los índices de las columnas dummy activas para esta fila
    dummies_activas = [col for col in alias_dummies_cols if row[col] == 1]
    if dummies_activas:
        # Filtra filas similares (al menos una dummy activa en común y price_num no nulo)
        similares = df_res[(df_res[dummies_activas] == 1).any(axis=1) & (df_res['price_num'].notna())]
        if not similares.empty:
            return similares['price_num'].mean()
    # Si no hay similares, retorna la media global
    return df_res['price_num'].mean()

# Aplica la función al DataFrame
df_res['price_num'] = df_res.apply(imputar_price_dummies, axis=1)

# Verifica que no queden nulos
print('Valores nulos en price_num tras imputación con dummies:', df_res['price_num'].isna().sum())


Valores nulos en price_num tras imputación con dummies: 0


Con ello, ya no tenemos valores nulos que puedan afectar a futuros análisis, además que se imputaron de manera inteligente. Hacemos una columna extra donde podamos identificar aquellos valores que fueron afectados y quede resgistro de ello

In [ ]:
# nueva columna que indique si el valor fue imputado o no

df_res['price_num_imputado'] = df_res['price'].isna() # Crear una nueva columna que indique si el valor fue imputado (True) o no (False)

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,rating,price,phone,coordinates.latitude,coordinates.longitude,...,alias_venues,alias_vietnamese,alias_waffles,alias_whiskeybars,alias_wine_bars,transaction_delivery,transaction_pickup,transaction_restaurant_reservation,price_num,price_num_imputado
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,4.4,$$$,13124926262,41.884193,-87.647946,...,0,0,0,0,0,1,0,0,3.000000,False
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,4.3,$$$,13124641744,41.890694,-87.624782,...,0,0,0,0,0,1,1,0,3.000000,False
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,4.5,$$,17737703427,41.917275,-87.698577,...,0,0,0,0,0,1,1,0,2.000000,False
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,4.1,$$,13127923502,41.881689,-87.625006,...,0,0,0,0,0,1,0,0,2.000000,False
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,4.7,$$$,13127963316,41.889390,-87.635240,...,0,0,0,0,0,1,0,0,3.000000,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,4.3,$$$,13127257811,41.887443,-87.617630,...,0,0,0,0,0,0,0,0,3.000000,False
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,4.3,$$,17736619028,41.931800,-87.704810,...,0,0,0,0,0,1,0,0,2.000000,False
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,4.1,NaN,0,41.896293,-87.667533,...,0,0,0,0,0,0,0,0,2.545455,True
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,4.1,$$,17734868525,41.917750,-87.695965,...,0,0,0,0,1,1,1,0,2.000000,False


Ya completado el Data Frame, hacemos el cambio de tipo de variable en la columna 'price_num' para que solo indique números enteros

In [ ]:
# Convertir a int en 'price_num' y hacer el redondeo hacia arriba si es más de 0.5

df_res['price_num'] = df_res['price_num'].round().astype(int) # Redondear los valores en 'price_num' y convertirlos a enteros

df_res # Mostramos el DataFrame actualizado para verificar los cambios

,id,alias,image_url,url,review_count,rating,price,phone,coordinates.latitude,coordinates.longitude,...,alias_venues,alias_vietnamese,alias_waffles,alias_whiskeybars,alias_wine_bars,transaction_delivery,transaction_pickup,transaction_restaurant_reservation,price_num,price_num_imputado
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,4.4,$$$,13124926262,41.884193,-87.647946,...,0,0,0,0,0,1,0,0,3,False
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,4.3,$$$,13124641744,41.890694,-87.624782,...,0,0,0,0,0,1,1,0,3,False
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,4.5,$$,17737703427,41.917275,-87.698577,...,0,0,0,0,0,1,1,0,2,False
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,4.1,$$,13127923502,41.881689,-87.625006,...,0,0,0,0,0,1,0,0,2,False
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,4.7,$$$,13127963316,41.889390,-87.635240,...,0,0,0,0,0,1,0,0,3,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,ak3FYZOsEWeiP6Pdp74YkQ,miru-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/_rz-Ub...,https://www.yelp.com/biz/miru-chicago?adjust_c...,278,4.3,$$$,13127257811,41.887443,-87.617630,...,0,0,0,0,0,0,0,0,3,False
196,lrwkR8IVohWQM551BUYcZQ,superkhana-international-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/0XZ9Y8...,https://www.yelp.com/biz/superkhana-internatio...,221,4.3,$$,17736619028,41.931800,-87.704810,...,0,0,0,0,0,1,0,0,2,False
197,Ki9_2R3o8kGtOBfJh1nrWg,hiro-izakaya-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/bYOO7R...,https://www.yelp.com/biz/hiro-izakaya-chicago?...,15,4.1,NaN,0,41.896293,-87.667533,...,0,0,0,0,0,0,0,0,3,True
198,fJkO3cfnNXrLIoYYGBX7_g,table-donkey-and-stick-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/PVHfLs...,https://www.yelp.com/biz/table-donkey-and-stic...,525,4.1,$$,17734868525,41.917750,-87.695965,...,0,0,0,0,1,1,1,0,2,False


In [ ]:
# Eliminar la columna 'price' si aún existe antes de exportar
df_res.drop(columns=[col for col in ['price'] if col in df_res.columns], inplace=True)
df_res.head() # Verificar que la columna fue eliminada

,id,alias,image_url,url,review_count,rating,phone,coordinates.latitude,coordinates.longitude,location.zip_code,...,alias_venues,alias_vietnamese,alias_waffles,alias_whiskeybars,alias_wine_bars,transaction_delivery,transaction_pickup,transaction_restaurant_reservation,price_num,price_num_imputado
0,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,10510,4.4,13124926262,41.884193,-87.647946,60607,...,0,0,0,0,0,1,0,0,3,False
1,boE4Ahsssqic7o5wQLI04w,the-purple-pig-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/rHHvhR...,https://www.yelp.com/biz/the-purple-pig-chicag...,8856,4.3,13124641744,41.890694,-87.624782,60611,...,0,0,0,0,0,1,1,0,3,False
2,gzhkdb6YoiFm5s3vriG1AA,gretel-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/-dxigk...,https://www.yelp.com/biz/gretel-chicago?adjust...,415,4.5,17737703427,41.917275,-87.698577,60647,...,0,0,0,0,0,1,1,0,2,False
3,riT822EnU7y_5eCuJsd9sA,cindys-rooftop-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/XLdHyZ...,https://www.yelp.com/biz/cindys-rooftop-chicag...,2717,4.1,13127923502,41.881689,-87.625006,60603,...,0,0,0,0,0,1,0,0,2,False
4,GZsrGq6H8CQ4YlGtE_Bm0Q,ciccio-mio-chicago-2,https://s3-media0.fl.yelpcdn.com/bphoto/eFIrEM...,https://www.yelp.com/biz/ciccio-mio-chicago-2?...,577,4.7,13127963316,41.889390,-87.635240,60654,...,0,0,0,0,0,1,0,0,3,False


Mostramos la información del Data Frame terminado para corroborar la limpieza y transformación exitosos

In [ ]:
df_res.info() # Verificar que no haya valores nulos y los tipos de datos sean correctos

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Columns: 117 entries, id to price_num_imputado
dtypes: bool(1), float64(3), int64(106), object(7)
memory usage: 181.6+ KB


In [ ]:
df_res.describe(include='all') # Revisamos el cambio de tipo de dato en la columna 'price_num'

,id,alias,image_url,url,review_count,rating,phone,coordinates.latitude,coordinates.longitude,location.zip_code,...,alias_venues,alias_vietnamese,alias_waffles,alias_whiskeybars,alias_wine_bars,transaction_delivery,transaction_pickup,transaction_restaurant_reservation,price_num,price_num_imputado
count,200,200,200,200,200.000000,200.000000,200,200.000000,200.000000,200,...,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200
unique,200,200,200,200,NaN,NaN,186,NaN,NaN,27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,qjnpkS8yZO8xcyEIy5OU9A,girl-and-the-goat-chicago,https://s3-media0.fl.yelpcdn.com/bphoto/ya6gjD...,https://www.yelp.com/biz/girl-and-the-goat-chi...,NaN,NaN,0,NaN,NaN,60622,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False
freq,1,1,1,1,NaN,NaN,14,NaN,NaN,36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,132
mean,NaN,NaN,NaN,NaN,811.705000,4.409500,NaN,41.906998,-87.659393,NaN,...,0.015000,0.010000,0.005000,0.005000,0.070000,0.700000,0.575000,0.065000,2.545000,NaN
std,NaN,NaN,NaN,NaN,1589.055178,0.216319,NaN,0.025849,0.027656,NaN,...,0.121857,0.099748,0.070711,0.070711,0.255787,0.459408,0.495584,0.247144,0.608132,NaN
min,NaN,NaN,NaN,NaN,4.000000,3.800000,NaN,41.848943,-87.728479,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,NaN
25%,NaN,NaN,NaN,NaN,84.750000,4.300000,NaN,41.890283,-87.677874,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,NaN
50%,NaN,NaN,NaN,NaN,274.000000,4.400000,NaN,41.903358,-87.655252,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,2.000000,NaN
75%,NaN,NaN,NaN,NaN,728.000000,4.500000,NaN,41.922381,-87.634322,NaN,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,0.000000,3.000000,NaN


Diagnóstico integral de valores nulos, vacíos y espacios en blanco
Con el objetivo de garantizar la calidad y consistencia de los datos, se realiza un diagnóstico exhaustivo sobre todas las columnas del DataFrame. Este análisis permite identificar la presencia de valores nulos, cadenas vacías y registros compuestos únicamente por espacios en blanco, especialmente en las columnas de tipo texto. De esta manera, se obtiene una visión clara de la cantidad y distribución de datos faltantes o potencialmente problemáticos, lo que facilita la toma de decisiones para su tratamiento en las siguientes etapas del preprocesamiento.

In [ ]:
# Diagnóstico robusto para todas las columnas: nulos, vacíos y solo espacios en blanco
import pandas as pd
resultados = []
for col in df_res.columns:
    if df_res[col].dtype == 'O':  # Solo columnas tipo objeto (texto)
        nulos = df_res[col].isnull().sum()
        vacios = (df_res[col] == '').sum()
        solo_espacios = (df_res[col].str.strip() == '').sum()
        total = (df_res[col].isnull() | (df_res[col].str.strip() == '')).sum()
        resultados.append({'columna': col, 'nulos': nulos, "vacios ('')": vacios, 'solo_espacios': solo_espacios, 'total_nulos_vacios_espacios': total})
    else:
        nulos = df_res[col].isnull().sum()
        resultados.append({'columna': col, 'nulos': nulos, "vacios ('')": 'N/A', 'solo_espacios': 'N/A', 'total_nulos_vacios_espacios': nulos})
diagnostico = pd.DataFrame(resultados)
display(diagnostico)

,columna,nulos,vacios (''),solo_espacios,total_nulos_vacios_espacios
0,id,0,0,0,0
1,alias,0,0,0,0
2,image_url,0,0,0,0
3,url,0,0,0,0
4,review_count,0,N/A,N/A,0
...,...,...,...,...,...
112,transaction_delivery,0,N/A,N/A,0
113,transaction_pickup,0,N/A,N/A,0
114,transaction_restaurant_reservation,0,N/A,N/A,0
115,price_num,0,N/A,N/A,0


Exportación del dataset limpio y procesado
Como etapa final del preprocesamiento, se procede a la exportación del DataFrame resultante, el cual ha sido depurado, transformado y enriquecido a lo largo de todo el análisis. Esta exportación permite disponer de un dataset listo para su utilización en análisis posteriores, modelado predictivo o integración con otras fuentes de datos, asegurando la calidad y consistencia de la información.

In [ ]:
# Exportar el DataFrame limpio y final a CSV (sin la columna 'price')
df_res.to_csv('rest_chicago.csv', index=False)
print('Archivo rest_chicago.csv exportado correctamente.')

Archivo rest_chicago.csv exportado correctamente.
